In [1]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [2]:
import datasets

In [3]:
dataset = datasets.load_dataset("datalama/exaone-3.5-macpie-test", split='train')

In [14]:
sim_ds = dataset.map(lambda x: {"simple_sanitized_instruction": x['instruction'].split("\n\n")[0]})

Map:   0%|          | 0/16400 [00:00<?, ? examples/s]

In [16]:
sim_ds[132]['simple_sanitized_instruction']

'Significantly revise and enhance the following paragraph to improve coherence, flow, validity, and engagement:'

In [ ]:
# Process conversations using HuggingFace datasets
processed_dataset = dataset.map(
    self.processor,
    batched=True,
    remove_columns=dataset.column_names,
    desc="Processing conversations"
)

In [ ]:
class ConversationProcessor(BaseProcessor):
    def __init__(self, config: ModelConfig):
        self.config = config
        
    def _process_single_conversation(self, conv_data: Dict[str, Any]) -> Optional[str]:
        if conv_data["language"] != self.config.language:
            return None
            
        processed_text = []
        for turn in conv_data["conversation_a"]:
            if turn["role"] != "user":
                continue
                
            content = turn["content"]
            if isinstance(content, list):
                processed_text.append(content[0])
            elif isinstance(content, str):
                processed_text.append(content)
            else:
                raise ValueError(f"Unknown content type: {type(content)}")
                
        conv = "\n".join(processed_text)
        conv = conv.replace("<|endoftext|>", "<| endoftext |>")
        
        if len(conv) <= 32:
            return None
            
        return conv[:self.config.max_conv_length]
    
    def __call__(self, examples: Dict[str, List], *extra_args) -> Dict[str, List]:
        """Process a batch of conversations."""
        processed_convs = []
        for i in range(len(examples["conversation_a"])):
            # Construct single example
            example = {key: examples[key][i] for key in examples.keys()}
            processed = self._process_single_conversation(example)
            processed_convs.append(processed if processed else "")
            
        return {"processed_text": processed_convs}

In [ ]:
datasets.load()

In [ ]:
# Load and process dataset
dataset = self._load_dataset(args.conv_file)
if args.first_n is not None:
    dataset = dataset.select(range(min(len(dataset), args.first_n)))

# Process conversations using HuggingFace datasets
processed_dataset = dataset.map(
    self.processor,
    batched=True,
    remove_columns=dataset.column_names,
    desc="Processing conversations"
)

# Filter out empty processed texts
processed_dataset = processed_dataset.filter(
    lambda x: bool(x['processed_text']),
    desc="Filtering empty texts"
)

# Generate embeddings
embeddings = self.embedding_generator.generate_embeddings(processed_dataset)

# Save processed data and embeddings
os.makedirs(args.output_dir, exist_ok=True)
processed_dataset.save_to_disk(f"{args.output_dir}/processed_dataset")
np.save(f"{args.output_dir}/embeddings.npy", embeddings)

---

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

model_kwargs = {"device": "cpu", "trust_remote_code": True}
encode_kwargs = {
    "task": "text-matching",
    "prompt_name": "text-matching",
}
embeddings = HuggingFaceEmbeddings(
    model_name="jinaai/jina-embeddings-v3",
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

text = "This is a test document."
query_result = embeddings.embed_query(text)

# show only the first 100 characters of the stringified vector
# print(str(query_result)[:100] + "...")

README.md:   0%|          | 0.00/734k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v3:
- custom_st.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [27]:
import numpy as np

In [30]:
tmp = embeddings.embed_query(['안녕', '디지몬'])
tmp.extend(embeddings.embed_documents(['안녕', '디지몬']))

In [32]:
len(tmp)

4

In [21]:
np.array([].extend(embeddings.embed_documents(['안녕', '디지몬']))

array([[ 0.03133342, -0.08279043,  0.02353187, ...,  0.02168752,
        -0.01921328,  0.0127812 ],
       [ 0.03441392, -0.00030601,  0.10557424, ...,  0.01044451,
        -0.05945493, -0.00494215]])

In [22]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("jinaai/jina-embeddings-v3", trust_remote_code=True, device="cpu")

task = "text-matching"
embeddings = model.encode(
    ['안녕', '디지몬'],
    task=task,
    prompt_name=task,
    normalize_embeddings=False
)

In [23]:
np.array(embeddings)

array([[ 0.03133342, -0.08279043,  0.02353187, ...,  0.02168752,
        -0.01921328,  0.0127812 ],
       [ 0.03441392, -0.00030601,  0.10557424, ...,  0.01044451,
        -0.05945493, -0.00494215]], dtype=float32)

---


In [1]:
from textanalyzer import Korean

In [9]:
def get_pos(doc):
    for token in doc:
        print(token.pos_)

In [2]:
nlp = Korean()

In [8]:
for token in nlp("안녕"):
    print(token.pos_)

INTJ


In [10]:
get_pos(nlp("11시 11분"))

NUM
NOUN
NUM
NOUN


In [11]:
from bertopic.representation import PartOfSpeech
from bertopic import BERTopic

In [12]:
# Create your representation model
representation_model = PartOfSpeech(nlp, [{"TAG":{"IN":["XR", "SL"]}}])

In [ ]:
representation_model.

In [ ]:


# Use the representation model in BERTopic on top of the default pipeline
topic_model = BERTopic(representation_model=representation_model)

In [ ]:
[
    [{'POS': 'ADJ'}, {'POS': 'NOUN'}],
    [{'POS': 'NOUN'}],
    [{'POS': 'ADJ'}]
]

In [13]:
import openai

In [20]:
from dotenv import load_dotenv, find_dotenv

In [21]:
load_dotenv(find_dotenv())

True

In [ ]:
import tiktoken

In [22]:
from openai import OpenAI
client = OpenAI()



SyncPage[Model](data=[Model(id='gpt-4o-audio-preview-2024-10-01', created=1727389042, object='model', owned_by='system'), Model(id='gpt-4o-mini-audio-preview', created=1734387424, object='model', owned_by='system'), Model(id='gpt-4o-mini', created=1721172741, object='model', owned_by='system'), Model(id='gpt-4o-realtime-preview', created=1727659998, object='model', owned_by='system'), Model(id='gpt-4o-realtime-preview-2024-10-01', created=1727131766, object='model', owned_by='system'), Model(id='gpt-4o-mini-audio-preview-2024-12-17', created=1734115920, object='model', owned_by='system'), Model(id='gpt-4o-mini-2024-07-18', created=1721172717, object='model', owned_by='system'), Model(id='gpt-4o-mini-realtime-preview', created=1734387380, object='model', owned_by='system'), Model(id='dall-e-2', created=1698798177, object='model', owned_by='system'), Model(id='gpt-4-1106-preview', created=1698957206, object='model', owned_by='system'), Model(id='gpt-4o-realtime-preview-2024-12-17', creat

In [25]:
set(map(lambda x: x.id, client.models.list().data))

{'babbage-002',
 'chatgpt-4o-latest',
 'dall-e-2',
 'dall-e-3',
 'davinci-002',
 'gpt-3.5-turbo',
 'gpt-3.5-turbo-0125',
 'gpt-3.5-turbo-1106',
 'gpt-3.5-turbo-16k',
 'gpt-3.5-turbo-16k-0613',
 'gpt-3.5-turbo-instruct',
 'gpt-3.5-turbo-instruct-0914',
 'gpt-4',
 'gpt-4-0125-preview',
 'gpt-4-0613',
 'gpt-4-1106-preview',
 'gpt-4-turbo',
 'gpt-4-turbo-2024-04-09',
 'gpt-4-turbo-preview',
 'gpt-4o',
 'gpt-4o-2024-05-13',
 'gpt-4o-2024-08-06',
 'gpt-4o-2024-11-20',
 'gpt-4o-audio-preview',
 'gpt-4o-audio-preview-2024-10-01',
 'gpt-4o-audio-preview-2024-12-17',
 'gpt-4o-mini',
 'gpt-4o-mini-2024-07-18',
 'gpt-4o-mini-audio-preview',
 'gpt-4o-mini-audio-preview-2024-12-17',
 'gpt-4o-mini-realtime-preview',
 'gpt-4o-mini-realtime-preview-2024-12-17',
 'gpt-4o-realtime-preview',
 'gpt-4o-realtime-preview-2024-10-01',
 'gpt-4o-realtime-preview-2024-12-17',
 'o1-mini',
 'o1-mini-2024-09-12',
 'o1-preview',
 'o1-preview-2024-09-12',
 'omni-moderation-2024-09-26',
 'omni-moderation-latest',
 'tex

In [18]:
openai.OpenAI.get_api_list()

TypeError: SyncAPIClient.get_api_list() missing 2 required positional arguments: 'self' and 'path'

In [26]:
import datasets

In [27]:
datasets.load_dataset("mteb/arena-results")

README.md:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/58 [00:00<?, ?it/s]

ValueError: Config name is missing.
Please pick one among the available configs: ['retrieval_battle', 'sts_battle', 'clustering_battle', 'sts_side_by_side', 'clustering_side_by_side', 'retrieval_side_by_side', 'clustering_individual', 'retrieval_individual', 'sts_individual']
Example of usage:
	`load_dataset('mteb/arena-results', 'retrieval_battle')`

---